In [6]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from scipy.optimize import minimize
from scipy.stats import norm

from pathlib import Path

# Display settings
pd.set_option("display.max_columns", None)
pd.set_option("display.float_format", "{:.6f}".format)

# Project directories
PROJECT_DIR = Path.cwd().parent
PROCESSED_DIR = PROJECT_DIR / "data" / "processed"
OUTPUT_DIR = PROJECT_DIR / "outputs" / "results"

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Load prepared asset returns
asset_returns_gbp = pd.read_csv(
    PROCESSED_DIR / "asset_returns_gbp.csv",
    index_col=0,
    parse_dates=True
)

asset_returns_usd = pd.read_csv(
    PROCESSED_DIR / "asset_returns_usd.csv",
    index_col=0,
    parse_dates=True
)

# Inspect the datasets
print("GBP returns:", asset_returns_gbp.shape)
print("USD returns:", asset_returns_usd.shape)

print("\nGBP date range:")
print(asset_returns_gbp.index.min(), "to", asset_returns_gbp.index.max())

print("\nAsset columns:")
print(asset_returns_gbp.columns.tolist())

display(asset_returns_gbp.head())

GBP returns: (1253, 5)
USD returns: (1253, 5)

GBP date range:
2021-10-13 00:00:00 to 2026-10-09 00:00:00

Asset columns:
['GLD', 'MSFT', 'QQQ', 'SPY', 'TLT']


,GLD,MSFT,QQQ,SPY,TLT
Date,,,,,
2021-10-13,0.018680,0.012591,0.008885,0.004470,0.010605
2021-10-14,-0.004183,0.014977,0.011682,0.010097,-0.002923
2021-10-15,-0.015866,0.004850,0.006328,0.007643,-0.005800
2021-10-18,-0.009202,0.003318,0.003249,-0.003798,-0.002081
2021-10-19,0.005050,0.004895,0.009411,0.009559,-0.011990


In [7]:
# Define the estimation and evaluation periods
TRAIN_START = "2021-10-12"
TRAIN_END = "2023-10-11"

OOS_START = "2023-10-12"
OOS_END = "2026-10-09"

# Split the GBP asset returns
gbp_train = asset_returns_gbp.loc[
    TRAIN_START:TRAIN_END
].copy()

gbp_oos = asset_returns_gbp.loc[
    OOS_START:OOS_END
].copy()

# Estimate the training-period covariance matrix
cov_gbp = gbp_train.cov()

# Define the minimum-variance optimisation function
def portfolio_variance(weights):
    return weights.T @ cov_gbp.to_numpy() @ weights

n_assets = len(gbp_train.columns)

initial_weights = np.repeat(1 / n_assets, n_assets)

constraints = ({
    "type": "eq",
    "fun": lambda weights: np.sum(weights) - 1
})

bounds = tuple((0, 1) for _ in range(n_assets))

# Run the optimisation
result = minimize(
    fun=portfolio_variance,
    x0=initial_weights,
    method="SLSQP",
    bounds=bounds,
    constraints=constraints,
    options={"maxiter": 1000, "ftol": 1e-12}
)

if not result.success:
    raise RuntimeError(result.message)

# Store the optimised weights
gbp_weights = pd.Series(
    result.x,
    index=gbp_train.columns,
    name="GBP Minimum Variance"
)

# Construct the equally weighted benchmark
equal_weights = pd.Series(
    1 / n_assets,
    index=gbp_train.columns,
    name="GBP Equal Weight"
)

# Construct out-of-sample portfolio returns
portfolio_returns = pd.DataFrame(index=gbp_oos.index)

portfolio_returns["GBP_Min_Variance"] = (
    gbp_oos @ gbp_weights
)

portfolio_returns["GBP_Equal_Weight"] = (
    gbp_oos @ equal_weights
)

# Inspect weights and returns
display(gbp_weights.to_frame())
display(portfolio_returns.head())

print("Training observations:", len(gbp_train))
print("Out-of-sample observations:", len(gbp_oos))
print("Portfolio return dimensions:", portfolio_returns.shape)

,GBP Minimum Variance
GLD,0.487654
MSFT,0.000000
QQQ,0.000000
SPY,0.266099
TLT,0.246247


,GBP_Min_Variance,GBP_Equal_Weight
Date,,
2023-10-12,-0.011798,-0.010718
2023-10-13,0.029887,0.015400
2023-10-16,-0.001120,0.005602
2023-10-17,-0.006541,-0.007561
2023-10-18,0.004035,-0.002656


Training observations: 502
Out-of-sample observations: 751
Portfolio return dimensions: (751, 2)


In [9]:
# Load the portfolio weights calculated in Notebook 03
portfolio_weights = pd.read_csv(
    OUTPUT_DIR / "portfolio_weights.csv",
    index_col=0
)

display(portfolio_weights)

,GBP Minimum Variance,0
GLD,0.487654,0.200000
MSFT,0.000000,0.200000
QQQ,0.000000,0.200000
SPY,0.266099,0.200000
TLT,0.246247,0.200000


In [10]:
### Historical Value at Risk

# Define the VaR estimation parameters

PORTFOLIO_VALUE = 100_000

CONFIDENCE_LEVELS = [0.95, 0.99]

WINDOW_SIZE = 250

print("Initial portfolio value: £", PORTFOLIO_VALUE)
print("Confidence levels:", CONFIDENCE_LEVELS)
print("Rolling window:", WINDOW_SIZE, "trading days")

# Combine training and out-of-sample asset returns

all_gbp_returns = pd.concat(
    [gbp_train, gbp_oos]
)

# Apply the fixed portfolio weights to all available dates

all_portfolio_returns = pd.DataFrame(index=all_gbp_returns.index)

all_portfolio_returns["GBP_Min_Variance"] = (
    all_gbp_returns @ gbp_weights
)

all_portfolio_returns["GBP_Equal_Weight"] = (
    all_gbp_returns @ equal_weights
)

display(all_portfolio_returns.head())
display(all_portfolio_returns.tail())

print("Total observations:", len(all_portfolio_returns))

Initial portfolio value: £ 100000
Confidence levels: [0.95, 0.99]
Rolling window: 250 trading days


,GBP_Min_Variance,GBP_Equal_Weight
Date,,
2021-10-13,0.012910,0.011046
2021-10-14,-0.000073,0.005930
2021-10-15,-0.007131,-0.000569
2021-10-18,-0.006010,-0.001703
2021-10-19,0.002054,0.003385


,GBP_Min_Variance,GBP_Equal_Weight
Date,,
2026-10-05,-0.003412,0.001517
2026-10-06,0.007352,0.007309
2026-10-07,-0.012806,-0.008113
2026-10-08,0.008672,0.001015
2026-10-09,0.008539,0.009304


Total observations: 1253


Let us start with one date 12 October 2023

In [11]:
# Select the first out-of-sample forecast date

first_forecast_date = gbp_oos.index[0]

# Retrieve the 250 observations before that date

historical_window = all_portfolio_returns.loc[
    all_portfolio_returns.index < first_forecast_date
].tail(WINDOW_SIZE)

# Display the historical observations

display(historical_window)

print("Forecast date:", first_forecast_date)
print("First historical observation:", historical_window.index.min())
print("Last historical observation:", historical_window.index.max())
print("Number of observations:", len(historical_window))

,GBP_Min_Variance,GBP_Equal_Weight
Date,,
2022-10-13,-0.011130,0.001612
2022-10-14,-0.031469,-0.036764
2022-10-17,0.012392,0.024609
2022-10-18,-0.004963,-0.004077
2022-10-19,-0.012062,-0.009318
...,...,...
2023-10-05,-0.007066,-0.006624
2023-10-06,-0.001449,0.005115
2023-10-09,0.013686,0.009485


Forecast date: 2023-10-12 00:00:00
First historical observation: 2022-10-13 00:00:00
Last historical observation: 2023-10-11 00:00:00
Number of observations: 250


In [12]:
# Calculate Historical VaR for the first forecast date

historical_var = {}

for portfolio in historical_window.columns:

    historical_var[portfolio] = {}

    for confidence in CONFIDENCE_LEVELS:

        tail_probability = 1 - confidence

        return_quantile = historical_window[portfolio].quantile(
            tail_probability
        )

        var_amount = -PORTFOLIO_VALUE * return_quantile

        historical_var[portfolio][confidence] = var_amount

# Convert the results into a DataFrame

historical_var_first_date = pd.DataFrame(historical_var)

display(historical_var_first_date)

,GBP_Min_Variance,GBP_Equal_Weight
0.950000,1440.413058,1542.542068
0.990000,1909.938576,2425.531611


In [13]:
### A function to calculate Historical VaR for any forecast date
def calculate_historical_var(
    historical_window,
    portfolio_value,
    confidence_levels
):

    results = {}

    for portfolio in historical_window.columns:

        results[portfolio] = {}

        for confidence in confidence_levels:

            tail_probability = 1 - confidence

            return_quantile = historical_window[
                portfolio
            ].quantile(tail_probability)

            var_amount = -portfolio_value * return_quantile

            results[portfolio][confidence] = var_amount

    return pd.DataFrame(results)

In [14]:
# Test Historical VaR for the first forecast date using the function
first_date_var = calculate_historical_var(
    historical_window=historical_window,
    portfolio_value=PORTFOLIO_VALUE,
    confidence_levels=CONFIDENCE_LEVELS
)

display(first_date_var)

,GBP_Min_Variance,GBP_Equal_Weight
0.950000,1440.413058,1542.542068
0.990000,1909.938576,2425.531611


In [15]:
### Calculate Historical VaR for all out-of-sample dates

# Create an empty list to store all rolling VaR forecasts

historical_var_results = []

# Loop through every out-of-sample date

for forecast_date in gbp_oos.index:

    # Select the 250 observations before the forecast date

    historical_window = all_portfolio_returns.loc[
        all_portfolio_returns.index < forecast_date
    ].tail(WINDOW_SIZE)

    # Check that we have the required number of observations

    if len(historical_window) < WINDOW_SIZE:
        continue

    # Calculate VaR for this forecast date

    daily_var = calculate_historical_var(
        historical_window=historical_window,
        portfolio_value=PORTFOLIO_VALUE,
        confidence_levels=CONFIDENCE_LEVELS
    )

    # Store the results alongside the forecast date

    daily_var = daily_var.T

    daily_var["Forecast_Date"] = forecast_date

    historical_var_results.append(daily_var)

# Combine all daily results into one DataFrame

historical_var_results = pd.concat(
    historical_var_results
)

# Set the forecast date as the DataFrame index

historical_var_results = historical_var_results.set_index(
    "Forecast_Date"
)

display(historical_var_results.head())
print("Total VaR forecasts:", len(historical_var_results))

,0.950000,0.990000
Forecast_Date,,
2023-10-12,1440.413058,1909.938576
2023-10-12,1542.542068,2425.531611
2023-10-13,1440.413058,1909.938576
2023-10-13,1542.542068,2425.531611
2023-10-16,1421.676435,1852.351822


Total VaR forecasts: 1502


In [17]:
# Give the existing index a meaningful name

historical_var_results = []

for forecast_date in gbp_oos.index:

    historical_window = all_portfolio_returns.loc[
        all_portfolio_returns.index < forecast_date
    ].tail(WINDOW_SIZE)

    if len(historical_window) < WINDOW_SIZE:
        continue

    daily_var = calculate_historical_var(
        historical_window=historical_window,
        portfolio_value=PORTFOLIO_VALUE,
        confidence_levels=CONFIDENCE_LEVELS
    )

    # Transpose so portfolios become rows
    daily_var = daily_var.T

    # Give the portfolio index an explicit column name
    daily_var.index.name = "Portfolio"

    # Convert the portfolio index into a normal column
    daily_var = daily_var.reset_index()

    # Attach the date of the VaR forecast
    daily_var["Forecast_Date"] = forecast_date

    historical_var_results.append(daily_var)


# Combine all daily results
historical_var_results = pd.concat(
    historical_var_results,
    ignore_index=True
)

# Arrange and rename the columns
historical_var_results = historical_var_results[
    ["Forecast_Date", "Portfolio", 0.95, 0.99]
]

historical_var_results = historical_var_results.rename(
    columns={
        0.95: "VaR_95",
        0.99: "VaR_99"
    }
)

display(historical_var_results.head(6))

,Forecast_Date,Portfolio,VaR_95,VaR_99
0,2023-10-12,GBP_Min_Variance,1440.413058,1909.938576
1,2023-10-12,GBP_Equal_Weight,1542.542068,2425.531611
2,2023-10-13,GBP_Min_Variance,1440.413058,1909.938576
3,2023-10-13,GBP_Equal_Weight,1542.542068,2425.531611
4,2023-10-16,GBP_Min_Variance,1421.676435,1852.351822
5,2023-10-16,GBP_Equal_Weight,1526.435013,2290.293178
